In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SalaryDetection") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Step 1: Initialize Spark
spark = SparkSession.builder \
    .appName("SalaryDetection") \
    .master("local[*]") \
    .getOrCreate()

# Step 2: Load data (example)
df = spark.read.csv("transactions.csv", header=True, inferSchema=True)

# Step 3: Window Spec
windowSpec = Window.partitionBy("customerId").orderBy("date")

# Step 4: Filter credits
df = df.filter(F.col("type") == "CREDIT")

# Step 5: Lag previous date
df = df.withColumn("prev_date", F.lag("date").over(windowSpec))

# Step 6: Interval calculation
df = df.withColumn(
    "interval_days",
    F.datediff(F.col("date"), F.col("prev_date"))
)

# Step 7: Aggregation
pattern_df = df.groupBy("customerId").agg(
    F.avg("interval_days").alias("avg_interval"),
    F.stddev("amount").alias("amount_std"),
    F.count("*").alias("txn_count")
)

# Step 8: Final classification
result = pattern_df.withColumn(
    "salary_pattern",
    F.when((F.col("avg_interval") >= 25) & (F.col("avg_interval") <= 35), "MONTHLY")
     .when((F.col("avg_interval") >= 12) & (F.col("avg_interval") <= 18), "BI_WEEKLY")
     .when((F.col("avg_interval") >= 5) & (F.col("avg_interval") <= 9), "WEEKLY")
     .otherwise("UNKNOWN")
).withColumn(
    "is_salary_account",
    F.when(
        (F.col("amount_std") < 5000) &
        (F.col("txn_count") >= 3) &
        (F.col("salary_pattern") != "UNKNOWN"),
        True
    ).otherwise(False)
)

result.show()

KeyboardInterrupt: 